## Demonstration of MultiObject and Clutter capabilities with Levy Processes

import stuff

In [ ]:
import numpy as np
from datetime import datetime, timedelta

from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel


simulate groundtruth

In [ ]:
# And the clock starts
start_time = datetime.now().replace(microsecond=0)

seed = 1 # Random seem for reproducibility

# Driving process parameters
mu_W = 0.02
sigma_W2 = 1
alpha = 1.4
c=10

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha)

driver_y = driver_x # Same driving process in both dimensions and sharing the same latents (jumps)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

from ordered_set import OrderedSet
np.random.seed(1991)

truths = OrderedSet()

num_steps = 40
num_objects=3 #must be <=5

timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))
truths.add(truth)

remaining_start_states = [[100, 1, 0, 1],[100, 1, 100, -1],[0, 1, 100, -1],[-100, 1, -100, -1]]
for state in remaining_start_states[:num_objects-1]:
    truth = GroundTruthPath([GroundTruthState(state, timestamp=timesteps[0])])
    for k in range(num_steps):
        truth.append(GroundTruthState(
            transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
            timestamp=timesteps[k+1]))
    truths.add(truth)


generate detections

In [ ]:
from scipy.stats import uniform

from stonesoup.types.detection import TrueDetection
from stonesoup.types.detection import Clutter
from stonesoup.models.measurement.linear import LinearGaussian

x_noise=10
y_noise=x_noise
measurement_model = LinearGaussian(
    ndim_state=4,
    mapping=(0, 2),
    noise_covar=np.array([[x_noise, 0],
                          [0, y_noise]])
    )
all_measurements = []

for k in range(num_steps):
    measurement_set = set()
    
    for truth in truths:
        # Generate actual detection from the state with a 10% chance that no detection is received.
        if np.random.rand() <= 0.9:
            measurement = measurement_model.function(truth[k], noise=True)
            measurement_set.add(TrueDetection(state_vector=measurement,
                                              groundtruth_path=truth,
                                              timestamp=truth[k].timestamp,
                                              measurement_model=measurement_model))

        # Generate clutter at this time-step
        truth_x = truth[k].state_vector[0]
        truth_y = truth[k].state_vector[2]
        for _ in range(np.random.randint(10)):
            x = uniform.rvs(-200, 400)
            y = uniform.rvs(-200, 400)
            measurement_set.add(Clutter(np.array([[x], [y]]), timestamp=truth[k].timestamp,
                                        measurement_model=measurement_model))
    all_measurements.append(measurement_set)

colors=['#00CC96','#FFA15A','#B6E880','#AB63FA','red']
from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=1)
plotter.colorway=colors
plotter.plot_ground_truths(truths, [0, 2])
# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])
# plotter.fig

generate 5 prior states

In [ ]:
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVectors

number_particles = 300

# Sample from the prior Gaussian distribution
states1 = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states2 = multivariate_normal.rvs(np.array(remaining_start_states[0]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states3 = multivariate_normal.rvs(np.array(remaining_start_states[1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states4 = multivariate_normal.rvs(np.array(remaining_start_states[2]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states5 = multivariate_normal.rvs(np.array(remaining_start_states[3]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)

covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)
# Create prior particle states.

prior1 = MarginalisedParticleState(
    state_vector=StateVectors(states1.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

prior2 = MarginalisedParticleState(
    state_vector=StateVectors(states2.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

prior3 = MarginalisedParticleState(
    state_vector=StateVectors(states3.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))
prior4 = MarginalisedParticleState(
    state_vector=StateVectors(states4.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))
prior5 = MarginalisedParticleState(
    state_vector=StateVectors(states5.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

generate tracking estimations

In [ ]:
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler 
from stonesoup.tracker.tests.conftest import data_associator
from stonesoup.updater.particle import MarginalisedParticleUpdater

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis, Euclidean
from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour, GNNWith2DAssignment
from stonesoup.dataassociator.probability import PDA
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

hypothesiser = DistanceHypothesiser(predictor, updater, measure=Mahalanobis(), missed_distance=5)
data_associator = GlobalNearestNeighbour(hypothesiser)
# data_associator = GNNWith2DAssignment(hypothesiser)


track1=Track()
track1.append(prior1)
track2=Track()
track2.append(prior2)
track3=Track()
track3.append(prior3)
track4=Track()
track4.append(prior4)
track5=Track()
track5.append(prior5)
tracks = [track1,track2,track3,track4,track5]

for n, measurements in enumerate(all_measurements):
    # Calculate all hypothesis pairs and associate the elements in the best subset to the tracks.
    hypotheses = data_associator.associate(tracks=tracks[:num_objects],
                                           detections=measurements,
                                           timestamp=start_time + timedelta(seconds=n))
    for j, track in enumerate(tracks[:num_objects]):
        hypothesis = hypotheses[track]
        if hypothesis.measurement:
            post = updater.update(hypothesis)
            track.append(post)
        else:  # When data associator says no detections are good enough, we'll keep the prediction
            track.append(hypothesis.prediction)
            print(f"  Track {j+1}: No Detection Assigned - Keeping Prediction {hypothesis.prediction.state_vector.mean}")
    print(f"iteration{n+1} of {len(all_measurements)}")


plot it all

In [ ]:
from stonesoup.plotter import AnimatedPlotterly, Plotterly, Dimension
from pathlib import Path

plotter = AnimatedPlotterly(timesteps, tail_length=0.5,height=800,width=1000)
all_measurements_list=list(all_measurements)
# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])

for i, track in enumerate(tracks[:num_objects]):
    plotter.plot_ground_truths(truths[i], [0, 2],line=dict(color=colors[i]),label=f'truth {i}')
    plotter.plot_tracks(track, [0, 2], uncertainty=True, line=dict(color=colors[i]) ,label=f'track {i}')
plotter.fig

plotter.fig.update_layout( 
    plot_bgcolor="white",  # Set background color to white
    xaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text="x position", font=dict(size=20)),  # Add large label
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text='y position', font=dict(size=20)),  # Add large label
    ),
    # legend=dict(
    #     font=dict(size=15),       # Make the legend font larger
    #     orientation='h',
    #     # xanchor="auto",         # Center the legend
    #     # yanchor="auto",           # Align the legend to the bottom of the plot
    #     bordercolor="Black",
    #     borderwidth=3,
    #     y=+0.45,                   # Position it above the graph
    #     # x=0.6                    # Center it horizontally
    # ),
    showlegend=False
)

In [ ]:
axis_label_list=["x","dx_dt","y","dy_dt"]
particle_plotter_dict = {}

for i,label in enumerate(axis_label_list):
    particle_plotter_dict[label]= Plotterly(autosize=False, width=1200,height=600, dimension=Dimension.ONE, axis_labels=[label])
    # if label =="x" or label=="y":
        # particle_plotter_dict[label].plot_measurements(all_measurements, [i],marker=dict(symbol="x",size=5))
    for j,truth in enumerate(truths):
        particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", truths_label=f"truth {j+1}",line=dict(width=1,color=colors[j]))
        particle_plotter_dict[label].plot_tracks(tracks[j], [i],mode="lines", track_label=f"track {j+1}", line=dict(width=1,color=colors[j]))
    particle_plotter_dict[label].fig.show()
    file_path = Path(rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TMR\multiobjectplot1D{label}.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    particle_plotter_dict[label].fig.write_html(str(file_path))

Test comparator from stonesoup website for gaussian tracker

In [ ]:
from datetime import datetime, timedelta
start_time = datetime.now().replace(microsecond=0)

from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, \
                                               ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from ordered_set import OrderedSet

np.random.seed(1991)

truths = OrderedSet()

transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.005),
                                                          ConstantVelocity(0.005)])

timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
for k in range(1, 21):
    timesteps.append(start_time+timedelta(seconds=k))
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))
truths.add(truth)

truth = GroundTruthPath([GroundTruthState([0, 1, 20, -1], timestamp=timesteps[0])])
for k in range(1, 21):
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))
_ = truths.add(truth)

from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truths, [0, 2])

from scipy.stats import uniform

from stonesoup.types.detection import TrueDetection
from stonesoup.types.detection import Clutter
from stonesoup.models.measurement.linear import LinearGaussian

measurement_model = LinearGaussian(
    ndim_state=4,
    mapping=(0, 2),
    noise_covar=np.array([[0.75, 0],
                          [0, 0.75]])
    )
all_measurements = []

for k in range(20):
    measurement_set = set()

    for truth in truths:
        # Generate actual detection from the state with a 10% chance that no detection is received.
        if np.random.rand() <= 0.9:
            measurement = measurement_model.function(truth[k], noise=True)
            measurement_set.add(TrueDetection(state_vector=measurement,
                                              groundtruth_path=truth,
                                              timestamp=truth[k].timestamp,
                                              measurement_model=measurement_model))

        # Generate clutter at this time-step
        truth_x = truth[k].state_vector[0]
        truth_y = truth[k].state_vector[2]
        for _ in range(np.random.randint(10)):
            x = uniform.rvs(truth_x - 10, 20)
            y = uniform.rvs(truth_y - 10, 20)
            measurement_set.add(Clutter(np.array([[x], [y]]), timestamp=truth[k].timestamp,
                                        measurement_model=measurement_model))
    all_measurements.append(measurement_set)

# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])



from stonesoup.predictor.kalman import KalmanPredictor
predictor = KalmanPredictor(transition_model)

from stonesoup.updater.kalman import KalmanUpdater
updater = KalmanUpdater(measurement_model)

from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
hypothesiser = DistanceHypothesiser(predictor, updater, measure=Mahalanobis(), missed_distance=3)


from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour
data_associator = GlobalNearestNeighbour(hypothesiser)

from stonesoup.types.state import GaussianState
prior1 = GaussianState([[0], [1], [0], [1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)
prior2 = GaussianState([[0], [1], [20], [-1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)

from stonesoup.types.track import Track
tracks = {Track([prior1]), Track([prior2])}

for n, measurements in enumerate(all_measurements):
    # Calculate all hypothesis pairs and associate the elements in the best subset to the tracks.
    hypotheses = data_associator.associate(tracks,
                                           measurements,
                                           start_time + timedelta(seconds=n))
    for track in tracks:
        hypothesis = hypotheses[track]
        if hypothesis.measurement:
            post = updater.update(hypothesis)
            track.append(post)
        else:  # When data associator says no detections are good enough, we'll keep the prediction
            track.append(hypothesis.prediction)


plotter.plot_tracks(tracks, [0, 2], uncertainty=True)
plotter.fig
